In [1]:
SYSTEM_PROMPT = """
Bạn là hệ thống trích xuất thông tin y khoa.

Nhiệm vụ:
Đọc bệnh án và trích xuất tất cả khái niệm y khoa.

Mỗi khái niệm phải có:

- text
- position
- type
- assertions
- candidates

Trong đó:

type phải là một trong:

TRIỆU_CHỨNG
TÊN_XÉT_NGHIỆM
KẾT_QUẢ_XÉT_NGHIỆM
CHẨN_ĐOÁN
THUỐC

assertions là list chỉ chứa các giá trị:

isNegated
isHistorical
isFamily

Nếu không có thì trả về []

candidates luôn trả về []

Không được giải thích.

Không được viết markdown.

Chỉ trả về JSON hợp lệ.

Lưu ý:
- "text" phải giống chính xác chuỗi xuất hiện trong văn bản.
- "position" là vị trí ký tự [start, end] trong văn bản.
"""

In [2]:
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/kangaonkaggle/viettel-ai-race-2026-data/Ontological Reasoning - 1st Round"
)

files = sorted(DATA_DIR.glob("*.txt"))

print(len(files))

100


In [3]:
text = files[0].read_text(encoding="utf8")

print(text[:1000])

1.  Tiền sử bệnh
    Thuốc trước khi nhập viện
    - metoprolol 25mg po bid
    - doxycycline cho viêm tuyến mồ hôi
    - atenolol (uống hôm nay)
    Các yếu tố nguy cơ liên quan
    - Theo lời bệnh nhân kể lại bệnh nhân có   Căng thẳng  nhiều trong công việc
    - Mất việc làm 8 ngày trước
    -  Một ngày người bệnh có thể uống hàng chục tách cà phê có caffeine
    -  và 1 tách cà phê không caffeine

2.  Tiền sử bệnh hiện tại
    Lý do nhập viện:  Bệnh nhân vào viện vì xuất hiện triệu chứng đánh trống ngực
    Thời điểm khởi phát triệu chứng: 
- Cách 10 ngày trước khi vào việc  bệnh nhân xuất hiện cảm giác đánh trống ngực
    Các triệu chứng hiện tại
    - đánh trống ngực
    - Khó thở nhẹ khó thở
    - cảm giác thắt chặt ngực vùng trước tim (khởi phát lúc 17 giờ)
    - Tăng đánh trống ngực (khởi phát lúc 17 giờ)
    - khó thở (khởi phát lúc 17 giờ)
    - Cảm thấy mệt mỏi nhiều khi gắng sức trong tuần qua
    - Cảm thấy mệt mỏi nhiều hơn sau khi luyện tập thể dục so với mọi ngày 
    

In [5]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": text
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=4096,
    do_sample=False,
    temperature=0.0
)

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print(response)

In [ ]:
import json

json_obj = json.loads(response)

with open("001.json", "w", encoding="utf8") as f:
    json.dump(
        json_obj,
        f,
        ensure_ascii=False,
        indent=2
    )

In [ ]:
from pathlib import Path
import json

OUT_DIR = Path("/kaggle/working/output")
OUT_DIR.mkdir(exist_ok=True)

for file in files:

    text = file.read_text(encoding="utf8")

    response = ask_qwen(text)

    try:
        obj = json.loads(response)

        with open(
            OUT_DIR / f"{file.stem}.json",
            "w",
            encoding="utf8"
        ) as f:

            json.dump(
                obj,
                f,
                ensure_ascii=False,
                indent=2
            )

    except Exception as e:
        print(file.name, e)